<a href="https://colab.research.google.com/github/AmarnathSharma10/recommendation-systems/blob/main/pratlip.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
chapters=pd.read_csv("/content/chapters.csv")
interactions=pd.read_csv("/content/interactions.csv")

In [6]:
chapters.head()

,chapter_id,chapter_sequence_no,book_id,author_id,published_date,tags
0,2812946,1,139726,66847,1990-03-22,Fantasy|Horror
1,4330764,2,139726,66847,1990-04-09,Fantasy|Young Adult|Literary Fiction
2,2664499,3,139726,66847,1990-04-07,Fantasy
3,2260666,4,139726,66847,1990-05-18,Literary Fiction|Fantasy
4,6069976,1,191772,62262,2008-07-30,Horror|Young Adult|Romance|Graphic Novel


In [7]:
interactions.head()

,user_id,chapter_id,book_id
0,user_2378720,5894067,444295
1,user_2321122,2532511,785684
2,user_2335775,6777764,999595
3,user_7906001,7366896,748410
4,user_9981689,7853186,418083


In [10]:
df = interactions.merge(chapters, on=["chapter_id", "book_id"])

In [11]:
user_book = df.groupby(['user_id', 'book_id'])['chapter_sequence_no'].max().reset_index()
total_chapters = chapters.groupby('book_id')['chapter_sequence_no'].max().reset_index()
total_chapters.rename(columns={'chapter_sequence_no': 'total_chapters'}, inplace=True)
user_book = user_book.merge(total_chapters, on='book_id')

user_book['progress'] = user_book['chapter_sequence_no'] / user_book['total_chapters']
user_book['weight'] = user_book['progress'] ** 2

In [12]:
book_meta = chapters.groupby('book_id').agg({
    'tags': 'first',
    'author_id': 'first',
    'published_date': 'first'
}).reset_index()


# Item-item book recommender
#
# **Task.** Given some books a user has interacted with, recommend other
# books they've also interacted with.
#
# **Framing.** This is static graph completion on the (user, book)
# bipartite graph. We do NOT assume any temporal order — the dataset has
# no timestamps and row order is unreliable across books. There is no
# "next" here; we predict held-out edges, not future ones.
#
# **Evaluation.** Leave-one-out per user. For each user with >= 2 books,
# hide one random book, feed the rest to the model, check whether the
# held-out book lands in the top-K recommendations. Report Hit@K and MRR.



In [13]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, diags
from collections import defaultdict

CHAPTERS_PATH     = "chapters.csv"
INTERACTIONS_PATH = "interactions.csv"
K                 = 10
SEED              = 42
N_EVAL_USERS      = 10_000


# 1. Load and inspect


In [14]:
chapters     = pd.read_csv(CHAPTERS_PATH)
interactions = pd.read_csv(INTERACTIONS_PATH)

print(f"chapters:     {len(chapters):>9,} rows  |  {chapters['book_id'].nunique():>7,} unique books")
print(f"interactions: {len(interactions):>9,} rows  |  {interactions['user_id'].nunique():>7,} unique users")

# Collapse to distinct (user, book) edges — that is the graph we model.
ub = interactions[["user_id", "book_id"]].drop_duplicates().reset_index(drop=True)
print(f"distinct (user, book) edges: {len(ub):,}")

books_per_user = ub.groupby("user_id").size()
print(f"books per user -> median: {books_per_user.median():.0f}, "
      f"mean: {books_per_user.mean():.2f}, "
      f"users with >=2 books: {(books_per_user >= 2).sum():,}")

chapters:        50,000 rows  |    9,575 unique books
interactions: 1,000,000 rows  |  149,803 unique users
distinct (user, book) edges: 999,520
books per user -> median: 6, mean: 6.67, users with >=2 books: 148,548


In [15]:
book_genres = (
    chapters.assign(g=chapters["tags"].fillna("").str.split("|"))
            .explode("g")
            .assign(g=lambda d: d["g"].str.strip())
            .query("g != ''")
            .groupby("book_id")["g"].apply(set)
            .to_dict()
)

In [16]:
rng = np.random.default_rng(SEED)
user_books = ub.groupby("user_id")["book_id"].apply(list).to_dict()

holdout = {}
history = {}
for u, blist in user_books.items():
    if len(blist) < 2:
        continue
    i = int(rng.integers(0, len(blist)))
    holdout[u] = blist[i]
    history[u] = [b for j, b in enumerate(blist) if j != i]

print(f"{len(holdout):,} users eligible for evaluation")

if N_EVAL_USERS is not None and len(holdout) > N_EVAL_USERS:
    sampled = rng.choice(list(holdout.keys()), size=N_EVAL_USERS, replace=False)
    holdout = {u: holdout[u] for u in sampled}
    print(f"subsampling to {len(holdout):,} users for eval")

148,548 users eligible for evaluation
subsampling to 10,000 users for eval


In [19]:
heldout_df = pd.DataFrame(
    [(u, b) for u, b in holdout.items()], columns=["user_id", "book_id"]
)
train_df = ub.merge(heldout_df, on=["user_id", "book_id"], how="left", indicator=True)
train_df = train_df[train_df["_merge"] == "left_only"][["user_id", "book_id"]]
print(f"training edges: {len(train_df):,}")

books = sorted(train_df["book_id"].unique())
users = sorted(train_df["user_id"].unique())
b2i   = {b: i for i, b in enumerate(books)}
u2i   = {u: i for i, u in enumerate(users)}

rows = train_df["user_id"].map(u2i).values
cols = train_df["book_id"].map(b2i).values
M    = csr_matrix(
    (np.ones(len(rows), dtype=np.float32), (rows, cols)),
    shape=(len(users), len(books)),
)

# Column-normalize so (M_norm.T @ M_norm)[i, j] = cosine(book_i, book_j)
col_norms = np.sqrt(np.asarray(M.multiply(M).sum(axis=0)).ravel())
col_norms[col_norms == 0] = 1.0
M_norm = (M @ diags(1.0 / col_norms)).tocsr()

book_pop = np.asarray(M.sum(axis=0)).ravel()
print(f"matrix {M.shape}, nnz={M.nnz:,}")

all_genres = sorted({g for gs in book_genres.values() for g in gs})
g2i = {g: i for i, g in enumerate(all_genres)}

g_rows, g_cols = [], []
for b, bi in b2i.items():
    for g in book_genres.get(b, ()):
        if g in g2i:
            g_rows.append(bi)
            g_cols.append(g2i[g])

G = csr_matrix(
    (np.ones(len(g_rows), dtype=np.float32), (g_rows, g_cols)),
    shape=(len(books), len(all_genres)),
)
g_norms = np.sqrt(np.asarray(G.multiply(G).sum(axis=1)).ravel())
g_norms[g_norms == 0] = 1.0
G_norm = (diags(1.0 / g_norms) @ G).tocsr()
print(f"book-genre matrix {G.shape}, {len(all_genres)} genres")


training edges: 989,520
matrix (149803, 9575), nnz=989,520
book-genre matrix (9575, 15), 15 genres


In [20]:
def _topk(score, history_books, k):
    for b in history_books:
        if b in b2i:
            score[b2i[b]] = -np.inf
    top = np.argpartition(-score, min(k, len(score) - 1))[:k]
    top = top[np.argsort(-score[top])]
    return [books[i] for i in top]

def rec_popularity(history_books, k=K):
    score = book_pop.astype(np.float32).copy()
    return _topk(score, history_books, k)

def _cf_scores(history_books):
    h = np.zeros(len(books), dtype=np.float32)
    for b in history_books:
        if b in b2i:
            h[b2i[b]] = 1.0
    # score = M_norm.T @ (M_norm @ h)   (sum of similarity cols over history)
    return M_norm.T @ (M_norm @ h)

def rec_cf(history_books, k=K):
    score = _cf_scores(history_books)
    if score.max() <= 0:
        return rec_popularity(history_books, k)
    return _topk(score, history_books, k)

def _content_scores(history_books):
    u = np.zeros(G_norm.shape[1], dtype=np.float32)
    n = 0
    for b in history_books:
        if b in b2i:
            u += np.asarray(G_norm[b2i[b]].todense()).ravel()
            n += 1
    if n:
        u /= n
        nrm = np.linalg.norm(u)
        if nrm > 0:
            u /= nrm
    return np.asarray(G_norm @ u).ravel()

def rec_content(history_books, k=K):
    score = _content_scores(history_books)
    if score.max() <= 0:
        return rec_popularity(history_books, k)
    return _topk(score, history_books, k)

def rec_hybrid(history_books, k=K):
    cf = _cf_scores(history_books)
    ct = _content_scores(history_books)
    if cf.max() > 0: cf = cf / cf.max()
    if ct.max() > 0: ct = ct / ct.max()
    # more history -> trust CF more
    alpha = min(0.80, 0.40 + 0.10 * len(history_books))
    score = alpha * cf + (1 - alpha) * ct
    if score.max() <= 0:
        return rec_popularity(history_books, k)
    return _topk(score, history_books, k)



In [21]:
def evaluate(recommender, name):
    hits, rrs = [], []
    for u, target in holdout.items():
        recs = recommender(history[u], k=K)
        if target in recs:
            hits.append(1)
            rrs.append(1.0 / (recs.index(target) + 1))
        else:
            hits.append(0)
            rrs.append(0.0)
    return {
        "model":     name,
        f"Hit@{K}":  round(float(np.mean(hits)), 4),
        "MRR":       round(float(np.mean(rrs)),  4),
        "n_users":   len(hits),
    }

results = pd.DataFrame([
    evaluate(rec_popularity, "popularity"),
    evaluate(rec_cf,         "item_item_cf"),
    evaluate(rec_content,    "content_genre"),
    evaluate(rec_hybrid,     "hybrid"),
])
print(results.to_string(index=False))


        model  Hit@10    MRR  n_users
   popularity  0.0078 0.0026    10000
 item_item_cf  0.0006 0.0002    10000
content_genre  0.0032 0.0012    10000
       hybrid  0.0048 0.0014    10000


In [22]:
def bucket(n):
    if n == 1: return "1"
    if n <= 3: return "2-3"
    if n <= 7: return "4-7"
    return "8+"

def evaluate_by_history(recommender, name):
    buckets = defaultdict(list)
    for u, target in holdout.items():
        recs = recommender(history[u], k=K)
        buckets[bucket(len(history[u]))].append(1 if target in recs else 0)
    return [
        {"model": name, "history": b, f"Hit@{K}": round(float(np.mean(v)), 4), "n": len(v)}
        for b, v in sorted(buckets.items(), key=lambda kv: ["1","2-3","4-7","8+"].index(kv[0]))
    ]

breakdown = pd.DataFrame(sum([
    evaluate_by_history(rec_popularity, "popularity"),
    evaluate_by_history(rec_cf,         "item_item_cf"),
    evaluate_by_history(rec_content,    "content_genre"),
    evaluate_by_history(rec_hybrid,     "hybrid"),
], []))
print(breakdown.to_string(index=False))

        model history  Hit@10    n
   popularity       1  0.0000  273
   popularity     2-3  0.0054 1672
   popularity     4-7  0.0096 5730
   popularity      8+  0.0060 2325
 item_item_cf       1  0.0000  273
 item_item_cf     2-3  0.0006 1672
 item_item_cf     4-7  0.0005 5730
 item_item_cf      8+  0.0009 2325
content_genre       1  0.0000  273
content_genre     2-3  0.0030 1672
content_genre     4-7  0.0035 5730
content_genre      8+  0.0030 2325
       hybrid       1  0.0000  273
       hybrid     2-3  0.0012 1672
       hybrid     4-7  0.0047 5730
       hybrid      8+  0.0082 2325
